# Phase 3 - Unsupervised ML Anomaly Detection
### Enterprise Deposit Data Quality & Anomaly Detection Pipeline

This notebook implements **Phase 3** of the *Field Guide* (`Field Guide.docx`) exactly as
specified, and runs it on the **cleaned outputs produced by Phase 2** for all three datasets
(HMDA Modified LAR, CFPB Consumer Complaints, SEC EDGAR Financial Statements).

**What Phase 3 does (per the Field Guide):**

1. **Feature scaling** - `RobustScaler` centres on the median and scales by the IQR, so a handful
   of giant deposits don't dominate the transform the way they would under `StandardScaler` -
   critical for heavy-tailed financial data.
2. **Model** - `IsolationForest` (**200 trees**, `contamination` = the expected **1 %** anomaly
   rate). Unsupervised, needs no labels, isolates outliers efficiently across high-dimensional
   financial features.
3. **Flags & scores** - `detect_anomalies()` appends `is_anomaly` (**1 = normal, -1 = anomalous**)
   and `anomaly_score` (negated decision function, so **higher = riskier**). `top_risk_audit()`
   returns the **top 1 %** highest-risk records as the manual-review queue.

**Scale note.** The cleaned files still contain millions of rows. Following standard practice for
Isolation Forest at scale, the model is **fitted on a representative sample** (configurable, default
up to ~1 M rows) and then used to **score every row in chunks**. The forest is unsupervised, so the
sample needs no labels; scoring is exact for all rows. Set `FIT_SAMPLE_ROWS = None` to fit on the
full cleaned table instead.

## 0 - Setup, paths and knobs

In [1]:
from __future__ import annotations
import os, json, logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

PROJECT_DIR = os.getcwd()
IN_DIR  = os.path.join(PROJECT_DIR, "phase2_cleaned_output")    # produced by the Phase 2 notebook
OUT_DIR = os.path.join(PROJECT_DIR, "phase3_anomaly_output")
os.makedirs(OUT_DIR, exist_ok=True)

# --- Field-Guide behaviour knobs -------------------------------------------
CONTAMINATION   = 0.01     # expected anomaly fraction (1%)
N_ESTIMATORS    = 200      # 200 trees, per the Field Guide
RANDOM_STATE    = 42
AUDIT_TOP_PCT   = 0.01     # export top 1% riskiest for human review
# --- scale knobs ------------------------------------------------------------
FIT_SAMPLE_ROWS = 1_000_000   # rows used to FIT the forest (None = fit on all rows)
CHUNK_SIZE      = 250_000     # rows per scoring block
SAMPLE_NROWS    = None        # cap rows READ from each cleaned file (None = all)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("phase3")

CLEANED = {
    "hmda": os.path.join(IN_DIR, "hmda_cleaned.csv"),
    "cfpb": os.path.join(IN_DIR, "cfpb_cleaned.csv"),
    "sec":  os.path.join(IN_DIR, "sec_cleaned.csv"),
}
for k, v in CLEANED.items():
    log.info("  %-6s -> %s (%s)", k, v, "FOUND" if os.path.exists(v) else "MISSING - run Phase 2 first")

16:48:29 | INFO    |   hmda   -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\phase2_cleaned_output\hmda_cleaned.csv (FOUND)
16:48:29 | INFO    |   cfpb   -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\phase2_cleaned_output\cfpb_cleaned.csv (FOUND)
16:48:29 | INFO    |   sec    -> C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\phase2_cleaned_output\sec_cleaned.csv (FOUND)


## 1 - Per-dataset model configuration
For each cleaned dataset we declare the **numeric feature columns** fed to the model, the
**segment** column (kept for the Tableau scatter/legend), and an **id** column. HMDA and SEC carry
real currency/numeric measures. CFPB has no currency, so we engineer numeric features from its
text/date fields (narrative length, response latency, missingness flags).

In [2]:
@dataclass
class ModelConfig:
    name: str
    path: str
    model_feature_cols: List[str] = field(default_factory=list)  # empty => auto-select numerics
    segment_col: Optional[str] = None
    id_col: Optional[str] = None
    keep_cols: List[str] = field(default_factory=list)           # extra cols to carry into outputs
    engineer: Optional[str] = None                               # name of a feature-engineering hook

MC_HMDA = ModelConfig(
    name="hmda", path=CLEANED["hmda"],
    model_feature_cols=["loan_amount","income","property_value","interest_rate","rate_spread",
                        "total_loan_costs","origination_charges","discount_points","lender_credits",
                        "loan_term"],
    segment_col="derived_loan_product_type", id_col="lei",
    keep_cols=["state","action_taken","loan_purpose","event_timestamp"],
)

MC_CFPB = ModelConfig(
    name="cfpb", path=CLEANED["cfpb"],
    model_feature_cols=[],            # built by the 'cfpb' engineer hook below
    segment_col="Product", id_col="Complaint ID",
    keep_cols=["Company","State","Submitted via","event_timestamp"],
    engineer="cfpb",
)

MC_SEC = ModelConfig(
    name="sec", path=CLEANED["sec"],
    model_feature_cols=["value","qtrs"],
    segment_col="form", id_col="cik",
    keep_cols=["name","tag","uom","event_timestamp"],
    engineer="sec",
)

In [3]:
def engineer_features(df: pd.DataFrame, mc: ModelConfig) -> pd.DataFrame:
    """Dataset-specific numeric feature engineering applied to each chunk."""
    df = df.copy()
    if mc.engineer == "cfpb":
        narr = df.get("Consumer complaint narrative")
        df["narrative_len"] = narr.fillna("").astype(str).str.len() if narr is not None else 0
        df["has_narrative"] = (narr.notna() & (narr.astype(str).str.len() > 0)).astype(int) if narr is not None else 0
        for c in ["Sub-issue","Sub-product","ZIP code","Tags"]:
            if c in df.columns:
                df[f"missing_{c.replace(' ','_').lower()}"] = df[c].isna().astype(int)
        # response latency (days) from received -> sent to company
        if "Date received" in df.columns and "Date sent to company" in df.columns:
            rec = pd.to_datetime(df["Date received"], errors="coerce", utc=True)
            snt = pd.to_datetime(df["Date sent to company"], errors="coerce", utc=True)
            df["days_to_company"] = (snt - rec).dt.total_seconds() / 86400.0
        if "Timely response?" in df.columns:
            df["timely_flag"] = (df["Timely response?"].astype(str).str.upper() == "YES").astype(int)
    elif mc.engineer == "sec":
        if "value" in df.columns:
            df["abs_value"] = df["value"].abs()
            df["log_abs_value"] = np.log1p(df["value"].abs())
        if "qtrs" in df.columns:
            df["qtrs"] = pd.to_numeric(df["qtrs"], errors="coerce")
    return df


def select_features(df: pd.DataFrame, mc: ModelConfig) -> List[str]:
    """Explicit feature list if given, else all numerics (minus leakage/id columns)."""
    if mc.model_feature_cols:
        feats = [c for c in mc.model_feature_cols if c in df.columns]
    else:
        feats = df.select_dtypes(include=[np.number]).columns.tolist()
    drop = {"is_anomaly", "anomaly_score"}
    return [c for c in feats if c not in drop]


# For CFPB the engineered features only exist after engineer_features(); declare their names.
CFPB_ENG_FEATURES = ["narrative_len","has_narrative","missing_sub-issue","missing_sub-product",
                     "missing_zip_code","missing_tags","days_to_company","timely_flag"]


## 2 - Field-Guide model functions
`detect_anomalies()` and `top_risk_audit()` are the Field Guide's functions. Here they are wrapped
so the forest is **fit once** (on a sample or the whole table) and then **scores every chunk**, which
keeps memory flat on multi-million-row files while producing exactly the same flags/scores the
Field Guide specifies.

In [4]:
def _chunk_reader(path: str):
    seen = 0
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE, low_memory=False):
        if SAMPLE_NROWS is not None and seen + len(chunk) > SAMPLE_NROWS:
            chunk = chunk.iloc[: SAMPLE_NROWS - seen]
        seen += len(chunk)
        yield chunk
        if SAMPLE_NROWS is not None and seen >= SAMPLE_NROWS:
            break


def _build_fit_sample(mc: ModelConfig) -> Tuple[pd.DataFrame, List[str]]:
    """Stream the file and collect a (capped) sample of engineered feature rows to fit on."""
    parts = []; feats = None; collected = 0
    for chunk in _chunk_reader(mc.path):
        chunk = engineer_features(chunk, mc)
        if feats is None:
            feats = CFPB_ENG_FEATURES if mc.engineer == "cfpb" else select_features(chunk, mc)
            feats = [c for c in feats if c in chunk.columns]
        block = chunk[feats].apply(pd.to_numeric, errors="coerce")
        parts.append(block)
        collected += len(block)
        if FIT_SAMPLE_ROWS is not None and collected >= FIT_SAMPLE_ROWS:
            break
    X = pd.concat(parts, ignore_index=True)
    if FIT_SAMPLE_ROWS is not None and len(X) > FIT_SAMPLE_ROWS:
        X = X.sample(FIT_SAMPLE_ROWS, random_state=RANDOM_STATE)
    return X, feats


def detect_anomalies_dataset(mc: ModelConfig) -> Dict:
    """PHASE 3 orchestrator for one dataset: fit RobustScaler+IsolationForest, score every row."""
    log.info("=" * 78); log.info("ANOMALY DETECTION: %s", mc.name.upper()); log.info("=" * 78)

    # ---- 1) fit scaler + forest on the (sampled) feature matrix -------------
    X_fit, feats = _build_fit_sample(mc)
    if not feats:
        raise ValueError(f"[{mc.name}] no numeric features available for anomaly detection.")
    medians = X_fit.median(numeric_only=True)
    X_fit = X_fit.fillna(medians)
    scaler = RobustScaler().fit(X_fit)
    model = IsolationForest(n_estimators=N_ESTIMATORS, contamination=CONTAMINATION,
                            random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(scaler.transform(X_fit))
    log.info("[%s] fitted IsolationForest on %d rows x %d features: %s",
             mc.name, len(X_fit), len(feats), feats)

    # ---- 2) score every row in chunks, write flagged output -----------------
    scored_path = os.path.join(OUT_DIR, f"{mc.name}_deposits_scored.csv")
    n_total = 0; n_anom = 0; wrote_header = False
    score_min = np.inf; score_max = -np.inf
    for chunk in _chunk_reader(mc.path):
        chunk = engineer_features(chunk, mc)
        X = chunk.reindex(columns=feats).apply(pd.to_numeric, errors="coerce").fillna(medians)
        Xs = scaler.transform(X)
        chunk["is_anomaly"] = model.predict(Xs)                  # 1 / -1
        chunk["anomaly_score"] = -model.decision_function(Xs)    # higher = riskier
        n_total += len(chunk); n_anom += int((chunk["is_anomaly"] == -1).sum())
        score_min = min(score_min, float(chunk["anomaly_score"].min()))
        score_max = max(score_max, float(chunk["anomaly_score"].max()))
        # keep outputs lean: id + segment + kept cols + features + flags
        keep = [c for c in ([mc.id_col, mc.segment_col] + mc.keep_cols + feats) if c and c in chunk.columns]
        keep = list(dict.fromkeys(keep)) + ["is_anomaly", "anomaly_score"]
        chunk[keep].to_csv(scored_path, mode=("w" if not wrote_header else "a"),
                           index=False, header=not wrote_header)
        wrote_header = True
    meta = {"dataset": mc.name, "features_used": feats, "rows": n_total,
            "n_anomalies": n_anom, "anomaly_rate": round(n_anom / max(n_total, 1), 4),
            "score_min": round(score_min, 4), "score_max": round(score_max, 4),
            "scored_csv": scored_path}
    log.info("[%s] flagged %d / %d rows (%.2f%%)", mc.name, n_anom, n_total,
             100 * meta["anomaly_rate"])
    return meta


## 3 - Top-1 % audit queue
A second streaming pass over the scored file keeps a running **top-N by `anomaly_score`** (N = 1 %
of rows) - the manual-review queue a commercial-banking analyst works, exported per the Field Guide.

In [5]:
def top_risk_audit_dataset(meta: Dict, mc: ModelConfig) -> str:
    """Stream the scored CSV and keep the top AUDIT_TOP_PCT riskiest records."""
    n_keep = max(1, int(meta["rows"] * AUDIT_TOP_PCT))
    top = None
    for chunk in pd.read_csv(meta["scored_csv"], chunksize=CHUNK_SIZE, low_memory=False):
        top = chunk if top is None else pd.concat([top, chunk], ignore_index=True)
        top = top.nlargest(n_keep, "anomaly_score")
    top = top.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
    top.insert(0, "audit_rank", range(1, len(top) + 1))
    out = os.path.join(OUT_DIR, f"{mc.name}_audit_queue_top1pct.csv")
    top.to_csv(out, index=False)
    log.info("[%s] audit queue: top %d records (%.1f%%) -> %s",
             mc.name, n_keep, 100 * AUDIT_TOP_PCT, os.path.basename(out))
    return out

## 4 - Run Phase 3 on all three datasets

In [6]:
results = {}
for mc in (MC_HMDA, MC_CFPB, MC_SEC):
    if not os.path.exists(mc.path):
        log.warning("SKIP %s - cleaned file missing (run Phase 2).", mc.name); continue
    meta = detect_anomalies_dataset(mc)
    meta["audit_queue_csv"] = top_risk_audit_dataset(meta, mc)
    results[mc.name] = meta

with open(os.path.join(OUT_DIR, "phase3_run_summary.json"), "w") as f:
    json.dump(results, f, indent=2, default=str)
print("Phase 3 complete. Outputs in:", OUT_DIR)

16:48:30 | INFO    | ==============================================================================
16:48:30 | INFO    | ANOMALY DETECTION: HMDA
16:48:30 | INFO    | ==============================================================================
16:48:48 | INFO    | [hmda] fitted IsolationForest on 1000000 rows x 10 features: ['loan_amount', 'income', 'property_value', 'interest_rate', 'rate_spread', 'total_loan_costs', 'origination_charges', 'discount_points', 'lender_credits', 'loan_term']
16:57:57 | INFO    | [hmda] flagged 134944 / 13468778 rows (1.00%)
16:58:48 | INFO    | [hmda] audit queue: top 134687 records (1.0%) -> hmda_audit_queue_top1pct.csv
16:58:48 | INFO    | ==============================================================================
16:58:48 | INFO    | ANOMALY DETECTION: CFPB
16:58:48 | INFO    | ==============================================================================
16:59:15 | INFO    | [cfpb] fitted IsolationForest on 1000000 rows x 8 features: ['narrative_

Phase 3 complete. Outputs in: C:\Users\Trong Khoi Van\Desktop\Denison\Banking Deposit Data ML Project\phase3_anomaly_output


In [7]:
# ---- Anomaly scorecard -----------------------------------------------------
rows = [{
    "dataset": n.upper(), "rows": m["rows"], "anomalies": m["n_anomalies"],
    "anomaly_rate_%": round(m["anomaly_rate"] * 100, 2),
    "max_anomaly_score": m["score_max"], "n_features": len(m["features_used"]),
} for n, m in results.items()]
pd.DataFrame(rows).set_index("dataset") if rows else "No results - run Phase 2 first."

,rows,anomalies,anomaly_rate_%,max_anomaly_score,n_features
dataset,,,,,
HMDA,13468778,134944,1.00,0.2119,10
CFPB,15694521,444022,2.83,0.1146,8
SEC,3690937,19082,0.52,0.1051,2


In [8]:
# ---- Peek at the HMDA audit queue (highest-risk records) -------------------
import os
q = os.path.join(OUT_DIR, "hmda_audit_queue_top1pct.csv")
pd.read_csv(q, nrows=10) if os.path.exists(q) else "Run the cells above first."

,audit_rank,lei,derived_loan_product_type,state,action_taken,loan_purpose,event_timestamp,loan_amount,income,property_value,interest_rate,rate_spread,total_loan_costs,origination_charges,discount_points,lender_credits,loan_term,is_anomaly,anomaly_score
0,1,54930091CYUK2SDZ6D37,CONVENTIONAL|FIRST_LIEN,CA,1,1,2025-01-01,3965000,4319.0,5295000.0,7.875,1.3760,52391.63,33728.38,31938.38,36642.37,360.0,-1,0.211875
1,2,2549004Z576VEJBK3N72,CONVENTIONAL|FIRST_LIEN,FL,1,1,2025-01-01,3325000,4724.0,4755000.0,8.499,2.1320,86967.00,68694.00,66500.00,13068.75,360.0,-1,0.211331
2,3,54930091CYUK2SDZ6D37,CONVENTIONAL|FIRST_LIEN,CA,1,31,2025-01-01,4215000,3298.0,8245000.0,7.125,0.9220,154433.22,143928.22,142138.22,57858.22,360.0,-1,0.211059
3,4,549300VZVN841I2ILS84,CONVENTIONAL|FIRST_LIEN,PA,1,1,2025-01-01,4005000,4163.0,7995000.0,8.375,1.8880,88940.25,68590.00,60000.00,9012.00,360.0,-1,0.208618
4,5,5493008PQSMR41Y70C36,CONVENTIONAL|FIRST_LIEN,FL,1,1,2025-01-01,11075000,4035.0,14765000.0,6.250,6.9770,106345.25,57729.00,56250.00,585.00,1.0,-1,0.205374
5,6,549300VZVN841I2ILS84,CONVENTIONAL|FIRST_LIEN,NY,1,1,2025-01-01,3745000,4345.0,4995000.0,8.375,2.3810,106266.97,95246.25,93656.25,7000.00,360.0,-1,0.203219
6,7,KB1H1DSPRFMYMCUFXT09,CONVENTIONAL|FIRST_LIEN,FL,1,1,2025-01-01,33955000,24731.0,48505000.0,5.125,-1.5134,151611.50,44736.50,2653.20,84920.00,360.0,-1,0.201468
7,8,54930048P8RWCQHQM310,CONVENTIONAL|FIRST_LIEN,NJ,1,1,2025-01-01,3005000,11749.0,6505000.0,7.750,0.9760,89990.58,76495.00,2653.20,57210.00,480.0,-1,0.200932
8,9,2549004Z576VEJBK3N72,CONVENTIONAL|FIRST_LIEN,AZ,1,31,2025-01-01,2705000,20941.0,4505000.0,8.250,1.6970,60850.57,57596.00,2653.20,55701.00,360.0,-1,0.200864
9,10,7H6GLXDRUGQFU57RNE97,CONVENTIONAL|FIRST_LIEN,CA,1,1,2025-01-01,4895000,1560.0,6525000.0,6.625,0.5100,88652.18,79600.18,77513.18,23300.00,360.0,-1,0.200713


## 5 - Outputs (Phase 4 / Tableau-ready)
For each dataset Phase 3 wrote into `phase3_anomaly_output/`:

* `<dataset>_deposits_scored.csv` - every record with `is_anomaly` (1 normal / -1 anomalous) and `anomaly_score` (higher = riskier).
* `<dataset>_audit_queue_top1pct.csv` - the top-1 % manual-review queue, ranked.
* `phase3_run_summary.json` - features used, counts and anomaly rates for all three runs.

These feed **Phase 4** directly: connect the scored CSVs in Tableau, set `event_timestamp` to
Date & Time and `anomaly_score` to a measure, and build the two-tab dashboard
(*Data Health* from Phase 2's health summaries, *Risk & Operational Anomalies* from these files).
The Field Guide's optional `pantab` step can convert any scored CSV to a native `.hyper` extract
for faster loads at full scale.